# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, following the Croissant FAIR schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset citation:**
Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

**URL:** https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset: {metadata.name}")
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below we print the record sets and for each record set, fields and columns with their `@id` values.

In [ ]:
# List all available record sets
record_sets = list(dataset.record_sets)

print("Available Record Sets (@id and name):\n-------------------------------")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# For each record set, list its fields and columns
print("\nRecord Set Fields and Columns:")
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) type: {field.data_type}")
    print("  Columns:")
    for column in rs.columns:
        print(f"    - {column.name} (@id: {column.id})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All record sets are referenced by their `@id`.

In [ ]:
# Prepare DataFrames for each record set, indexed by record set @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # The Rows API yields each record as a dict
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"No records found for record set: {rs_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set @id: {rs_id} - columns: {list(df.columns)}")

# Pick the first non-empty record set for demonstration
for rs_id in record_set_ids:
    if rs_id in dataframes:
        print(f"\nSample rows for record set {rs_id}:")
        display(dataframes[rs_id].head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We'll select a numeric field (e.g., 'MW' for molecular weight or similar field) to demonstrate filtering, normalization, and grouping.

All operations reference data by their Croissant schema `@id`.

In [ ]:
# Identify a numeric field from the field list by @id
# For demonstration, let's assume there is a field with @id 'MW' (Molecular Weight)
# We'll also choose a group field, e.g. 'description' (protein description), if available

# Find a record set and numeric field
numeric_field_id = None
group_field_id = None
rs_id = None

# Try to pick a record set with at least one float/integer field
for rs in record_sets:
    for field in rs.fields:
        if field.data_type in ['Float', 'Integer', 'Number']:
            numeric_field_id = field.id
            rs_id = rs.id
            # Try to find a non-numeric field for grouping
            for f in rs.fields:
                if f.id != numeric_field_id and f.data_type not in ['Float', 'Integer', 'Number']:
                    group_field_id = f.id
                    break
            break
    if numeric_field_id:
        break

if not (numeric_field_id and rs_id):
    print("No suitable record set with numeric field found.")
else:
    print(f"Using record set: {rs_id}")
    print(f"Numeric field: {numeric_field_id}")
    print(f"Group field: {group_field_id}")

df = dataframes[rs_id]

# Filtering
threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
if numeric_field_id in filtered_df.columns and filtered_df[numeric_field_id].std() > 0:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}, mean of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution for the numeric field
if numeric_field_id and rs_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id is available, plot the group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        group_means = group_means.sort_values(numeric_field_id, ascending=False).head(10)
        plt.figure(figsize=(10,4))
        sns.barplot(x=numeric_field_id, y=group_field_id, data=group_means, orient='h')
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 Croissant-formatted mass spectrometry dataset, explored available record sets and fields using their `@id`s, extracted tabular data, and performed exploratory processing and visualization. This workflow demonstrates how to make data-centric analyses more robust and reproducible using the `mlcroissant` library and semantic references.
